In [1]:
import os
import copy

import numpy as np
import yt
from yt.frontends.ramses.field_handlers import RTFieldFileHandler
import matplotlib.pyplot as plt
import scipy

from merlin_spectra.emission import EmissionLineInterpolator
from merlin_spectra import galaxy_visualization

import constants
import silmaril


In [2]:
#PATH TO SIMULATION
filename = constants.SIMULATION_PATH

#PATH TO Merlin
merlin_path = constants.MERLIN_PATH

#PATH TO silmaril/data
data = "/mnt/sim/Merlin/drivers/data/"


lines=["H1_6562.80A","O1_1304.86A","O1_6300.30A","O2_3728.80A","O2_3726.10A",
       "O3_1660.81A","O3_1666.15A","O3_4363.21A","O3_4958.91A","O3_5006.84A", 
       "He2_1640.41A","C2_1335.66A","C3_1906.68A","C3_1908.73A","C4_1549.00A",
       "Mg2_2795.53A","Mg2_2802.71A","Ne3_3868.76A","Ne3_3967.47A",
       "N5_1238.82A",
       "N5_1242.80A","N4_1486.50A","N3_1749.67A","S2_6716.44A","S2_6730.82A"]

wavelengths=[6562.80, 1304.86, 6300.30, 3728.80, 3726.10, 1660.81, 1666.15,
             4363.21, 4958.91, 5006.84, 1640.41, 1335.66,
             1906.68, 1908.73, 1549.00, 2795.53, 2802.71, 3868.76,
             3967.47, 1238.82, 1242.80, 1486.50, 1749.67, 6716.44, 6730.82]

cell_fields = [
    "Density",
    "x-velocity",
    "y-velocity",
    "z-velocity",
    "Pressure",
    "Metallicity",
    "xHI",
    "xHII",
    "xHeII",
    "xHeIII",
]

epf = [
    ("particle_family", "b"),
    ("particle_tag", "b"),
    ("particle_birth_epoch", "d"),
    ("particle_metallicity", "d"),
]

In [3]:
# Ionization Parameter Field
# Based on photon densities in bins 2-4
# Don't include bin 1 -> Lyman Werner non-ionizing
def _ion_param(field, data):
    p = RTFieldFileHandler.get_rt_parameters(ds).copy()
    p.update(ds.parameters)

    cgs_c = 2.99792458e10     #light velocity

    # Convert to physical photon number density in cm^-3
    pd_2 = data['ramses-rt','Photon_density_2']*p["unit_pf"]/cgs_c
    pd_3 = data['ramses-rt','Photon_density_3']*p["unit_pf"]/cgs_c
    pd_4 = data['ramses-rt','Photon_density_4']*p["unit_pf"]/cgs_c

    photon = pd_2 + pd_3 + pd_4

    return photon/data['gas', 'number_density']


def _my_temperature(field, data):
    #y(i): abundance per hydrogen atom
    XH_RAMSES=0.76 #defined by RAMSES in cooling_module.f90
    YHE_RAMSES=0.24 #defined by RAMSES in cooling_module.f90
    mH_RAMSES=yt.YTArray(1.6600000e-24,"g") #defined by RAMSES in cooling_module.f90
    kB_RAMSES=yt.YTArray(1.3806200e-16,"erg/K") #defined by RAMSES in cooling_module.f90

    dn=data["ramses","Density"].in_cgs()
    pr=data["ramses","Pressure"].in_cgs()
    yHI=data["ramses","xHI"]
    yHII=data["ramses","xHII"]
    yHe = YHE_RAMSES*0.25/XH_RAMSES
    yHeII=data["ramses","xHeII"]*yHe
    yHeIII=data["ramses","xHeIII"]*yHe
    yH2=1.-yHI-yHII
    yel=yHII+yHeII+2*yHeIII
    mu=(yHI+yHII+2.*yH2 + 4.*yHe) / (yHI+yHII+yH2 + yHe + yel)
    return pr/dn * mu * mH_RAMSES / kB_RAMSES


#number density of hydrogen atoms
def _my_H_nuclei_density(field, data):
    dn=data["ramses","Density"].in_cgs()
    XH_RAMSES=0.76 #defined by RAMSES in cooling_module.f90
    YHE_RAMSES=0.24 #defined by RAMSES in cooling_module.f90
    mH_RAMSES=yt.YTArray(1.6600000e-24,"g") #defined by RAMSES in cooling_module.f90

    return dn*XH_RAMSES/mH_RAMSES


def _OII_ratio(field, data):
    # TODO lum or flux?
    #return data['gas', 'flux_O2_3728.80A']/data['gas', 'flux_O2_3726.10A']
    flux1 = data['gas', 'flux_O2_3728.80A']
    flux2 = data['gas', 'flux_O2_3726.10A']

    flux2 = np.where(flux2 < 1e-30, 1e-30, flux2)

    ratio = flux1 / flux2

    return ratio


def _pressure(field, data):
    if 'hydro_thermal_pressure' in dir(ds.fields.ramses): # and 
        #'Pressure' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_thermal_pressure']


def _xHI(field, data):
    if 'hydro_xHI' in dir(ds.fields.ramses): # and \
        #'xHI' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHI']


def _xHII(field, data):
    if 'hydro_xHII' in dir(ds.fields.ramses): # and \
        #'xHII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHII']


def _xHeII(field, data):
    if 'hydro_xHeII' in dir(ds.fields.ramses): # and \
        #'xHeII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHeII']


def _xHeIII(field, data):
    if 'hydro_xHeIII' in dir(ds.fields.ramses): # and \
        #'xHeIII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHeIII']

In [4]:
def get_luminosity(self, line):
    '''
    Return function for derived luminosity field of each line.
    The Cloudy flux is obtained assuming a gas cloud of height = 1 cm,
    Returns flux values erg s^-1 c^-2.
    Multiply the flux at each cell by the volume of the cell
    to obtain the intrinsic luminosity.

    line (str): desired emission line from field
    '''

    def _luminosity(field, data):
        return data['gas', 'flux_' + line]*data['gas', 'volume']
    return copy.deepcopy(_luminosity)


In [5]:
ds = yt.load(filename, extra_particle_fields=epf)

yt : [INFO     ] 2026-04-01 17:49:55,202 Parameters: current_time              = 0.3604448649237178 Gyr
yt : [INFO     ] 2026-04-01 17:49:55,203 Parameters: domain_dimensions         = [64 64 64]
yt : [INFO     ] 2026-04-01 17:49:55,206 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-04-01 17:49:55,208 Parameters: domain_right_edge         = [1. 1. 1.]
yt : [INFO     ] 2026-04-01 17:49:55,210 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2026-04-01 17:49:55,212 Parameters: current_redshift          = 12.171087046255657
yt : [INFO     ] 2026-04-01 17:49:55,214 Parameters: omega_lambda              = 0.685000002384186
yt : [INFO     ] 2026-04-01 17:49:55,216 Parameters: omega_matter              = 0.314999997615814
yt : [INFO     ] 2026-04-01 17:49:55,218 Parameters: omega_radiation           = 0.0
yt : [INFO     ] 2026-04-01 17:49:55,219 Parameters: hubble_constant           = 0.674000015258789


In [6]:

'''
-------------------------------------------------------------------------------
Load Simulation Data
Add Derived Fields
-------------------------------------------------------------------------------
'''


ds.add_field(
    ("gas","number_density"),
    function=_my_H_nuclei_density,
    sampling_type="cell",
    units="1/cm**3",
    force_override=True
)

ds.add_field(
    ("ramses","Pressure"),
    function=_pressure,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHI"),
    function=_xHI,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHII"),
    function=_xHII,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHeII"),
    function=_xHeII,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHeIII"),
    function=_xHeIII,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("gas","my_temperature"),
    function=_my_temperature,
    sampling_type="cell",
    # TODO units
    #units="K",
    #units="K*cm**3/erg",
    units='K*cm*dyn/erg',
    force_override=True
)

# Ionization parameter
ds.add_field(
    ('gas', 'ion_param'),
    function=_ion_param,
    sampling_type="cell",
    units="cm**3",
    force_override=True
)

ds.add_field(
    ("gas","my_H_nuclei_density"),
    function=_my_H_nuclei_density,
    sampling_type="cell",
    units="1/cm**3",
    force_override=True
)


# Normalize by Density Squared Flag
dens_normalized = False
if dens_normalized: 
    flux_units = '1/cm**6'
    lum_units = '1/cm**3'
else:
    flux_units = '1'
    lum_units = 'cm**3'

# Instance of EmissionLineInterpolator for line list at filename
#line_list = os.path.join(os.getcwd(), 'data/linelist.dat')
# TODO alter for zaratan
line_list = os.path.join(merlin_path + '/src/merlin_spectra/linelists/linelist2.dat')
emission_interpolator = EmissionLineInterpolator(line_list, lines)

# Add flux and luminosity fields for all lines in the list
for i, line in enumerate(lines):
    ds.add_field(
        ('gas', 'flux_' + line),
        function=emission_interpolator.get_line_emission(
            i, dens_normalized=dens_normalized
        ),
        sampling_type='cell',
        units=flux_units,
        force_override=True,
    )

    ds.add_field(
        ('gas', 'luminosity_' + line),
        function=emission_interpolator.get_luminosity(lines[i]),
        sampling_type='cell',
        units=lum_units,
        force_override=True,
    )


ds.add_field(
    ("gas","OII_ratio"),
    function=_OII_ratio,
    sampling_type="cell",
    units="1",
    force_override=True
)
# TODO


ad = ds.all_data()
print(ds.field_list)
print(ds.derived_field_list)

yt : [WARNING  ] 2026-04-01 17:49:55,443 This output has no cooling fields
yt : [INFO     ] 2026-04-01 17:49:56,478 Adding particle_type: DM
yt : [INFO     ] 2026-04-01 17:49:56,498 Adding particle_type: star
yt : [INFO     ] 2026-04-01 17:49:56,516 Adding particle_type: cloud
yt : [INFO     ] 2026-04-01 17:49:56,533 Adding particle_type: dust
yt : [INFO     ] 2026-04-01 17:49:56,546 Adding particle_type: star_tracer
yt : [INFO     ] 2026-04-01 17:49:56,557 Adding particle_type: cloud_tracer
yt : [INFO     ] 2026-04-01 17:49:56,569 Adding particle_type: dust_tracer
yt : [INFO     ] 2026-04-01 17:49:56,583 Adding particle_type: gas_tracer
yt : [WARNING  ] 2026-04-01 17:49:56,613 Field ('ramses', 'Pressure') already exists. To override use `force_override=True`.


minU=-9.0, maxU=2.0, stepU=0.5, minN=-4.0, maxN=7.0, stepN=0.5, minT=1.0, maxT=8.0, stepT=0.2
Line List Shape = (25, 19044)
23 23 36
[('gravity', 'Potential'), ('gravity', 'x-acceleration'), ('gravity', 'y-acceleration'), ('gravity', 'z-acceleration'), ('io', 'particle_birth_epoch'), ('io', 'particle_family'), ('io', 'particle_identity'), ('io', 'particle_mass'), ('io', 'particle_metallicity'), ('io', 'particle_position_x'), ('io', 'particle_position_y'), ('io', 'particle_position_z'), ('io', 'particle_refinement_level'), ('io', 'particle_tag'), ('io', 'particle_velocity_x'), ('io', 'particle_velocity_y'), ('io', 'particle_velocity_z'), ('nbody', 'particle_mass'), ('nbody', 'particle_position_x'), ('nbody', 'particle_position_y'), ('nbody', 'particle_position_z'), ('nbody', 'particle_velocity_x'), ('nbody', 'particle_velocity_y'), ('nbody', 'particle_velocity_z'), ('ramses', 'Density'), ('ramses', 'Metallicity'), ('ramses', 'Pressure'), ('ramses', 'hydro_refinement-param'), ('ramses', 

In [7]:
x1 = ad["star", "particle_position_x"].in_units("pc")
y1 = ad["star", "particle_position_y"].in_units("pc")
z1 = ad["star", "particle_position_z"].in_units("pc")

center_pc = (np.mean(x1), np.mean(y1), np.mean(z1))

viz = galaxy_visualization.VisualizationManager(filename, lines, wavelengths)
star_ctr = viz.star_center(ad)
sp = ds.sphere(star_ctr, (3000, "pc"))
sp_lum = ds.sphere(star_ctr, (10, 'kpc'))
width = (1500, 'pc')

yt : [INFO     ] 2026-04-01 17:49:57,794 Identified   384/  384 intersecting domains (  385 through hilbert key indexing)


Filename = C:/Users/jonat/source/repos/UMD/GEMS/ricotti/output_00273/info_00273.txt
File Directory = C:/Users/jonat/source/repos/UMD/GEMS/ricotti/output_00273
Output File = output_00273
Simulation Run = 00273
Analysis Directory = analysis/output_00273_analysis


In [8]:
sc = yt.create_scene(sp_lum, field=("gas", "density"), )

In [15]:
sc.camera.focus = star_ctr
sc.camera.position = star_ctr + [5000, 5000, 5000]

In [10]:
source = sc[0]
source.set_log(True)

tf = source.transfer_function
tf.clear()
tf.add_layers(5, colormap="viridis")

yt : [INFO     ] 2026-04-01 17:50:00,642 Creating transfer function
yt : [INFO     ] 2026-04-01 17:50:00,644 Calculating data bounds. This may take a while. Set the TransferFunctionHelper.bounds to avoid this.
yt : [INFO     ] 2026-04-01 17:50:00,665 Identified   384/  384 intersecting domains (  385 through hilbert key indexing)


In [20]:
sc.save("sphere_render.png", sigma_clip=4.0)

yt : [WARNING  ] 2026-04-01 17:55:24,486 No previously rendered image found, rendering now.
yt : [INFO     ] 2026-04-01 17:55:24,487 Rendering scene (Can take a while).
yt : [INFO     ] 2026-04-01 17:55:24,489 Creating volume
yt : [INFO     ] 2026-04-01 17:55:25,048 Identified   162/  384 intersecting domains (  385 through hilbert key indexing)
yt : [INFO     ] 2026-04-01 17:55:28,775 Identified 1 intersecting domains
yt : [WARNING  ] 2026-04-01 17:55:36,164 RAMSESDomainSubset (info_00273): , base_region=YTRegion (info_00273): , center=[5.98491924e+24 5.98491924e+24 5.98491924e+24] cm, left_edge=[0. 0. 0.] cm, right_edge=[1.19698385e+25 1.19698385e+25 1.19698385e+25] cm, domain=RAMSESDomainFile: 1, ds=info_00273.retrieve_ghost_zones was called with the `smoothed` argument set to True. This is not supported, ignoring it.
yt : [INFO     ] 2026-04-01 17:55:36,169 Identified 1 intersecting domains
yt : [WARNING  ] 2026-04-01 17:55:38,652 RAMSESDomainSubset (info_00273): , base_region=YTRe

In [19]:
sp = ds.sphere(star_ctr, (3000, "pc"))

sc = yt.create_scene(ds, field=("gas", "density"))
source = sc[0]

# restrict to sphere

source.set_log(True)

tf = yt.ColorTransferFunction((1e-30, 1e-24))
tf.clear()

tf.add_layers(
    10,
    colormap="inferno"
)

# projection-like behavior
tf.grey_opacity = True

source.transfer_function = tf